In [3]:
import torch, transformers, datasets
from tqdm import tqdm
from sklearn.metrics import confusion_matrix

In [4]:
dataset = datasets.load_dataset("cornell-movie-review-data/rotten_tomatoes")

In [5]:
dataset = datasets.DatasetDict({
    "train": dataset["train"].shuffle(seed=42).select(range(2000)),
    "validation": dataset["validation"].shuffle(seed=42).select(range(500)),
    "test": dataset["test"].shuffle(seed=42).select(range(1000))
})

In [6]:
for split in ["train", "validation", "test"]:
    print(f"\n=== {split.upper()} SAMPLE ===")
    print(dataset[split][0])


=== TRAIN SAMPLE ===
{'text': '. . . plays like somebody spliced random moments of a chris rock routine into what is otherwise a cliche-riddled but self-serious spy thriller .', 'label': 0}

=== VALIDATION SAMPLE ===
{'text': "the filmmaker's heart is in the right place . . .", 'label': 1}

=== TEST SAMPLE ===
{'text': 'unpretentious , charming , quirky , original', 'label': 1}


In [7]:
pretrained_name = "bert-base-uncased"
tokenizer = transformers.BertTokenizerFast.from_pretrained(pretrained_name)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

e:\studies_master\deep-learning-in-text-processing\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\KOMP\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [8]:
MAX_LEN = 128
def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN
    )
dataset = dataset.map(tokenize_batch, batched=True)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [9]:
dataset["train"][0]

{'text': '. . . plays like somebody spliced random moments of a chris rock routine into what is otherwise a cliche-riddled but self-serious spy thriller .',
 'label': 0,
 'input_ids': [101,
  1012,
  1012,
  1012,
  3248,
  2066,
  8307,
  11867,
  13231,
  2094,
  6721,
  5312,
  1997,
  1037,
  3782,
  2600,
  9410,
  2046,
  2054,
  2003,
  4728,
  1037,
  18856,
  17322,
  1011,
  21834,
  2094,
  2021,
  2969,
  1011,
  3809,
  8645,
  10874,
  1012,
  102,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0],
 'token_type_ids': [0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,

In [10]:
print(tokenizer.decode(101))
print(tokenizer.decode(1))
print(tokenizer.decode(21398))
print(tokenizer.decode(2000))

[CLS]
[unused0]
##working
to


In [11]:
columns = ["input_ids", "attention_mask", "label"]

for split in ["train", "validation", "test"]:
    dataset[split].set_format(type="torch", columns=columns)
dataset["train"][0]

{'label': tensor(0),
 'input_ids': tensor([  101,  1012,  1012,  1012,  3248,  2066,  8307, 11867, 13231,  2094,
          6721,  5312,  1997,  1037,  3782,  2600,  9410,  2046,  2054,  2003,
          4728,  1037, 18856, 17322,  1011, 21834,  2094,  2021,  2969,  1011,
          3809,  8645, 10874,  1012,   102,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,   

In [12]:
train_loader = torch.utils.data.DataLoader(dataset["train"], batch_size=8, shuffle=True)
val_loader = torch.utils.data.DataLoader(dataset["validation"], batch_size=16)
test_loader = torch.utils.data.DataLoader(dataset["test"], batch_size=16)

In [13]:
model = transformers.BertForSequenceClassification.from_pretrained(
    pretrained_name,
    num_labels=2
)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model.to(device)

cuda


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [15]:
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
num_epochs = 3
num_training_steps = num_epochs * len(train_loader)
num_warmup_steps = int(0.1 * num_training_steps)
scheduler = transformers.get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)

In [16]:
best_val_loss = float("inf")
patience = 2
trigger_times = 0

for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0
    for batch in tqdm(train_loader, desc=f"Training Epoch {epoch+1}/{num_epochs}"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss
        total_loss += loss.item()
        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
    avg_train_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}:= Average Training Loss: {avg_train_loss:.4f}")

    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Validation Epoch {epoch+1}/{num_epochs}"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            loss = outputs.loss
            val_loss += loss.item()

            logits = outputs.logits
            predictions = torch.argmax(logits, dim=-1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)
    avg_val_loss = val_loss / len(val_loader)
    val_accuracy = correct / total
    print(f"Epoch {epoch+1}:= Average Validation Loss: {avg_val_loss:.4f}, Validation Accuracy: {val_accuracy:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        trigger_times = 0
        model.save_pretrained("best_model")
        tokenizer.save_pretrained("best_model")
    else:
        trigger_times += 1
        if trigger_times >= patience:
            print("Early stopping triggered.")
            break

Training Epoch 1/3: 100%|██████████| 250/250 [00:13<00:00, 19.16it/s]


Epoch 1:= Average Training Loss: 1.0270


Validation Epoch 1/3: 100%|██████████| 32/32 [00:00<00:00, 45.64it/s]


Epoch 1:= Average Validation Loss: 0.3538, Validation Accuracy: 0.8580


Training Epoch 2/3: 100%|██████████| 250/250 [00:12<00:00, 19.23it/s]


...

Epoch 2:= Average Validation Loss: 0.3293, Validation Accuracy: 0.8660


Training Epoch 3/3: 100%|██████████| 250/250 [00:12<00:00, 19.49it/s]


Epoch 3:= Average Training Loss: 0.2160


Validation Epoch 3/3: 100%|██████████| 32/32 [00:00<00:00, 45.34it/s]

Epoch 3:= Average Validation Loss: 0.3830, Validation Accuracy: 0.8700


In [17]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=-1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

acc = sum([p==l for p,l in zip(all_preds, all_labels)]) / len(all_labels)
print(f"Test Accuracy: {acc:.4f}")

cm = confusion_matrix(all_labels, all_preds)
print("Confusion Matrix:")
print(cm)

for i in range(5):
    print(f"Text: {tokenizer.decode(dataset['test'][i]['input_ids'], skip_special_tokens=True)}")
    print(f"Label: {dataset['test'][i]['label']}, Pred: {all_preds[i]}")
    print("-"*50)

Test Accuracy: 0.8270
Confusion Matrix:
[[436  65]
 [108 391]]
Text: unpretentious, charming, quirky, original
Label: 1, Pred: 1
--------------------------------------------------
Text: a film really has to be exceptional to justify a three hour running time, and this isn ' t.
Label: 0, Pred: 0
--------------------------------------------------
Text: working from a surprisingly sensitive script co - written by gianni romoli... ozpetek avoids most of the pitfalls you ' d expect in such a potentially sudsy set - up.
Label: 1, Pred: 1
--------------------------------------------------
Text: it may not be particularly innovative, but the film ' s crisp, unaffected style and air of gentle longing make it unexpectedly rewarding.
Label: 1, Pred: 1
--------------------------------------------------
Text: such a premise is ripe for all manner of lunacy, but kaufman and gondry rarely seem sure of where it should go.
Label: 0, Pred: 0
--------------------------------------------------
